In [1]:
import pandas as pd
import re
import ast
from pathlib import Path

# CONFIGURATION

INPUT_FILE = "../raw/ecommerce-products-dataset.csv"                
OUTPUT_FILE = "../processed/products.csv"                           

TARGET_PRODUCTS = 1000

# Conversion rate:
# 1 USD = 280 PKR
PKR_PER_USD = 280

# Keeps the random sample reproducible
RANDOM_STATE = 42

In [2]:
print("LOADING DATASET")

input_path = Path(INPUT_FILE)

if not input_path.exists():
    raise FileNotFoundError(
        f"Could not find '{INPUT_FILE}'. "
        f"Make sure the CSV is in the same folder as the notebook."
    )

df = pd.read_csv(INPUT_FILE)

print(f"Original rows: {len(df):,}")
print(f"Original columns: {len(df.columns)}")

print("\nOriginal columns:")
print(df.columns.tolist())

LOADING DATASET
Original rows: 8,479
Original columns: 11

Original columns:
['product_name', 'product_price', 'product_image', 'product_link', 'product_store', 'product_category', 'product_ratings', 'rating_count', 'description', 'date', 'source_file']


In [4]:
# main categiry mapping
MAIN_CATEGORY_MAP = {
    "mobilephones": "Mobile Phones",
    "mobile-phones": "Mobile Phones",
    "mobile phones": "Mobile Phones",

    "laptops": "Laptops",
    "laptop": "Laptops",

    "tablets": "Tablets",
    "tablet": "Tablets",

    "smartwatches": "Smart Watches",
    "smart-watches": "Smart Watches",
    "smart watches": "Smart Watches",

    "earbuds": "Earbuds",
    "earbud": "Earbuds",
}

In [5]:
# ACCESSORY DETECTION KEYWORDS

ACCESSORY_CATEGORY_KEYWORDS = [
    "mobile accessories",
    "mobile-accessories",
    "phone accessories",
    "phone-accessories",
    "laptop accessories",
    "laptop-accessories",
    "tablet accessories",
    "tablet-accessories",
    "watch accessories",
    "watch-accessories",
]


ACCESSORY_TITLE_KEYWORDS = [
    # Phone cases / covers
    "iphone case",
    "phone case",
    "mobile case",
    "back cover",
    "phone cover",
    "mobile cover",

    # Screen / camera protection

    "screen protector",
    "screen protection",
    "screen shield",
    "tempered glass",
    "camera lens protector",
    "camera protector",
    "lens protector",

    # Cables

    "charging cable",
    "data cable",
    "usb cable",
    "type-c cable",
    "type c cable",
    "lightning cable",
    "micro usb cable",

    # Chargers / adapters
  
    "wall charger",
    "car charger",
    "charging adapter",
    "power adapter",
    "travel adapter",

    # Power banks
   
    "power bank",
    "powerbank",
   
    # Phone holders / mounts
  
    "phone holder",
    "mobile holder",
    "phone mount",
    "mobile mount",
    "car mount",

    # Laptop accessories

    "laptop bag",
    "laptop sleeve",
    "laptop stand",
    "laptop cooler",
    "cooling pad",
    "mouse pad",

    # Other accessories
  
    "keyboard cover",
    "replacement battery",
    "replacement screen",
]

In [88]:
# Define known brands
KNOWN_BRANDS = [
    # Mobile phone brands
    "Samsung",
    "Xiaomi",
    "Vivo",
    "Oppo",
    "Infinix",
    "Tecno",
    "Realme",
    "Apple",
    "Itel",
    "Motorola",
    "OnePlus",
    "Google",
    "Sparx",
    "Nokia",

    # Laptop brands
    "HP",
    "Dell",
    "Lenovo",
    "Asus",
    "Acer",
    "MSI",
    "Microsoft",
    "Razer",

    # Tablet / drawing tablet brands
    "Amazon",
    "Wacom",
    "XP Pen",
    "Huawei",
    "Sony",
    "Dany",
    "Pen Power",

    # Smart watch brands
    "Mibro",
    "Amazfit",
    "Fitbit",
    "Kospet",
    "Kieslect",
    "Haylou",
    "Yolo",
    "Zero",
    "HainoTeko",
    "Amband",

    # Earbud / audio brands
    "JBL",
    "A4Tech",
    "Soundpeats",
    "Faster",
    "Beats",
    "UNIQ",
]

In [89]:
# define brand allies
BRAND_ALIASES = {
    "mi": "Xiaomi",
    "mmi": "Xiaomi",

    "iphone": "Apple",
    "ipad": "Apple",
    "ipads": "Apple",
    "airpods": "Apple",

    "amazon fire": "Amazon",
    "amazon kindle": "Amazon",

    "amazfit": "Amazfit",

    "hainteko": "HainoTeko",
    "hainoeko": "HainoTeko",

    "itel": "Itel",
}

In [90]:
def clean_text(value):
    """
    Clean missing values and normalize whitespace.
    """

    if pd.isna(value):
        return ""

    value = str(value)

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()

In [91]:
def is_accessory(row):
    """
    Return True only when the product appears to be
    an actual accessory rather than the main device.
    """

    category = str(
        row.get("product_category", "")
    ).strip().lower()

    title = str(
        row.get("title", row.get("product_name", ""))
    ).strip().lower()

    # Category-based detection

    for keyword in ACCESSORY_CATEGORY_KEYWORDS:

        if keyword in category:
            return True
        
    # Title-based detection

    for keyword in ACCESSORY_TITLE_KEYWORDS:

        if keyword in title:
            return True

    return False

In [92]:
DEVICE_OPENERS = [
    "apple watch", "samsung galaxy watch", "galaxy watch",
    "xiaomi mi smart band", "xiaomi mi band", "mi band", "mi smart band",
    "kospet", "amazfit", "haylou", "huawei watch", "garmin",
    "iphone", "ipad", "macbook", "galaxy tab", "galaxy s", "galaxy a",
    "redmi", "airpods",
]

ACCESSORY_NOUN_PATTERN = re.compile(
    r"\b(case|cover|sleeve|skin|screen protector|dock|adapter|cable|"
    r"charger|strap|band)\b",
    re.IGNORECASE,
)

FOR_PATTERN = re.compile(r"\bfor\b", re.IGNORECASE)


def is_accessory_by_pattern(row):
    title = str(
        row.get("title", row.get("product_name", ""))
    ).strip().lower()

    if not title:
        return False

    if any(title.startswith(opener) for opener in DEVICE_OPENERS):
        return False

    return bool(ACCESSORY_NOUN_PATTERN.search(title))

In [93]:
def get_main_category(category):
    """
    Normalize the main product category.
    """

    category = clean_text(category)

    if not category:
        return None

    first_part = (
        category
        .split(",")[0]
        .strip()
        .lower()
    )

    first_part = re.sub(
        r"\s+",
        " ",
        first_part
    )

    return MAIN_CATEGORY_MAP.get(
        first_part
    )

In [94]:
def normalize_brand(brand):
    """
    Normalize brand names and aliases.
    """

    brand = clean_text(brand)

    if not brand:
        return None

    brand_lower = brand.lower()

    # Alias lookup
    if brand_lower in BRAND_ALIASES:

        return BRAND_ALIASES[
            brand_lower
        ]

    # Known brand lookup
    for known_brand in KNOWN_BRANDS:

        if (
            brand_lower
            == known_brand.lower()
        ):
            return known_brand

    return brand

In [95]:
def extract_brand_from_category(category):
    """
    Extract a known brand from the category field.

    Example:
        Laptops, Apple, Moshi
        -> Apple

        MobilePhones, Samsung
        -> Samsung

    The function prioritizes known brands rather than
    blindly taking the second category value.
    """

    category = clean_text(category)

    if not category:
        return None

    parts = [
        clean_text(part)
        for part in category.split(",")
        if clean_text(part)
    ]

    # Search every category component for a known brand


    for part in parts:

        normalized_part = normalize_brand(part)

        for known_brand in KNOWN_BRANDS:

            if (
                normalized_part
                and
                normalized_part.lower()
                == known_brand.lower()
            ):
                return known_brand

    # Handle cases where a category component contains
    # multiple words, e.g. "Apple Moshi"

    category_text = " ".join(parts).lower()

    # Longest brands first
    sorted_brands = sorted(
        KNOWN_BRANDS,
        key=len,
        reverse=True
    )

    for brand in sorted_brands:

        pattern = (
            r"\b"
            + re.escape(brand.lower())
            + r"\b"
        )

        if re.search(
            pattern,
            category_text
        ):
            return brand

    return None

    


In [96]:
def extract_brand_from_title(title):
    """
    If the category does not contain a brand,
    search for a known brand in the product title.
    """

    title = clean_text(title)

    if not title:
        return None

    title_lower = title.lower()

    # Longest brands first
    sorted_brands = sorted(
        KNOWN_BRANDS,
        key=len,
        reverse=True
    )

    for brand in sorted_brands:

        pattern = (
            r"\b"
            + re.escape(
                brand.lower()
            )
            + r"\b"
        )

        if re.search(
            pattern,
            title_lower
        ):

            return brand

    return None

In [97]:
def extract_brand(category, title):
    """
    Extract brand using category first,
    then product title as fallback.
    """

    brand = extract_brand_from_category(
        category
    )

    if brand:
        return brand

    return extract_brand_from_title(
        title
    )

In [98]:
def get_subcategory(
    main_category,
    title,
    description
):
    """
    Create meaningful product-type subcategories
    based primarily on the product title.
    """

    title = clean_text(title).lower()
    description = clean_text(description).lower()

    text = f"{title} {description}"

    # MOBILE PHONES

    if main_category == "Mobile Phones":

        return "Smartphones"

    # LAPTOPS

    if main_category == "Laptops":

        if any(keyword in text for keyword in [
            "gaming",
            "rog",
            "alienware",
            "omen",
            "predator",
            "legion",
            "nitro",
            "tuf gaming",
            "razer blade"
        ]):
            return "Gaming Laptops"

        if any(keyword in text for keyword in [
            "elitebook",
            "probook",
            "thinkpad",
            "latitude",
            "precision",
            "surface laptop"
        ]):
            return "Business Laptops"

        if any(keyword in text for keyword in [
            "macbook",
            "macbook air",
            "macbook pro"
        ]):
            return "MacBooks"

        return "Laptops"

    # TABLETS

    if main_category == "Tablets":

        if any(keyword in text for keyword in [
            "ipad",
            "ipad air",
            "ipad pro",
            "ipad mini"
        ]):
            return "iPads"

        if any(keyword in text for keyword in [
            "drawing tablet",
            "graphics tablet",
            "pen tablet",
            "wacom",
            "xp-pen"
        ]):
            return "Drawing Tablets"

        return "Android Tablets"

    # SMART WATCHES

    if main_category == "Smart Watches":

        if any(keyword in text for keyword in [
            "apple watch",
            "galaxy watch"
        ]):
            return "Premium Smart Watches"

        return "Smart Watches"

    # EARBUDS

    if main_category == "Earbuds":

        if any(keyword in text for keyword in [
            "airpods"
        ]):
            return "AirPods"

        if any(keyword in text for keyword in [
            "tws",
            "true wireless",
            "wireless earbuds",
            "wireless earbud"
        ]):
            return "Wireless Earbuds"

        return "Earbuds"

    # FALLBACK

    return "Other"

In [99]:

print("REMOVING ACCESSORIES")

# changed 
df["is_accessory"] = df.apply(
    lambda row: is_accessory(row) or is_accessory_by_pattern(row),
    axis=1
)


accessories_found = (
    df["is_accessory"].sum()
)

print(
    f"Accessories detected: "
    f"{accessories_found:,}"
)


# Keep only actual products
df = df[
    ~df["is_accessory"]
].copy()


# Remove helper column
df.drop(
    columns=["is_accessory"],
    inplace=True
)


print(
    f"Products remaining: "
    f"{len(df):,}"
)

REMOVING ACCESSORIES
Accessories detected: 1,007
Products remaining: 7,472


In [101]:
print("CREATING MAIN CATEGORY")

df["main_category"] = (
    df["product_category"]
    .apply(get_main_category)
)

unknown_categories = (
    df["main_category"]
    .isna()
    .sum()
)

print(
    f"Unknown categories: "
    f"{unknown_categories:,}"
)

df = df.dropna(
    subset=["main_category"]
).copy()

print(
    f"Rows remaining: "
    f"{len(df):,}"
)

print("\nCategory distribution:")

print(
    df["main_category"]
    .value_counts()
)

CREATING MAIN CATEGORY
Unknown categories: 0
Rows remaining: 7,207

Category distribution:
main_category
Mobile Phones    3901
Tablets          1428
Laptops          1114
Smart Watches     501
Earbuds           263
Name: count, dtype: int64


In [102]:
print("RENAMING COLUMNS")

df = df.rename(
    columns={
        "product_name": "title",
        "product_image": "img_url",
        "product_ratings": "rating"
    }
)

print(df.columns.tolist())

RENAMING COLUMNS
['title', 'product_price', 'img_url', 'product_link', 'product_store', 'product_category', 'rating', 'rating_count', 'description', 'date', 'source_file', 'main_category']


In [103]:
print("CLEANING TEXT FIELDS")

df["title"] = (
    df["title"]
    .apply(clean_text)
)

df["img_url"] = (
    df["img_url"]
    .apply(clean_text)
)

CLEANING TEXT FIELDS


In [104]:
print("REMOVING KNOWN-BROKEN IMAGE HOST")
print(
    "A live spot-check (50-row sample) found every sampled URL from "
    "cdn.homeshopping.pk failing with HTTP 525 (SSL Handshake Failed) "
    "— a domain-wide infrastructure issue on that host's end, confirmed "
    "manually in a browser too, not a scattered dead-link problem. "
    "Full per-row validation across the whole dataset proved too slow "
    "in practice (15+ minutes for a partial pass), so this host is "
    "excluded outright rather than checked URL-by-URL."
)

before = len(df)
df = df[~df["img_url"].str.contains("cdn.homeshopping.pk", na=False)].copy()
removed = before - len(df)
print(f"Removed {removed:,} rows from cdn.homeshopping.pk")
print(f"Rows remaining: {len(df):,}")

REMOVING KNOWN-BROKEN IMAGE HOST
A live spot-check (50-row sample) found every sampled URL from cdn.homeshopping.pk failing with HTTP 525 (SSL Handshake Failed) — a domain-wide infrastructure issue on that host's end, confirmed manually in a browser too, not a scattered dead-link problem. Full per-row validation across the whole dataset proved too slow in practice (15+ minutes for a partial pass), so this host is excluded outright rather than checked URL-by-URL.
Removed 1,405 rows from cdn.homeshopping.pk
Rows remaining: 5,802


In [105]:
def clean_description(description):
    """
    Convert dictionary-like or list-like descriptions
    into readable text.
    """

    if pd.isna(description):
        return ""

    description = clean_text(
        description
    )

    if not description:
        return ""

    try:

        parsed = ast.literal_eval(
            description
        )

        if isinstance(parsed, dict):

            text_parts = []

            for key, value in parsed.items():

                key = clean_text(key)
                value = clean_text(value)

                if key and value:

                    text_parts.append(
                        f"{key} {value}"
                    )

            return ". ".join(
                text_parts
            )

        elif isinstance(parsed, list):

            # HomeShopping-style scraped page fragments — often
            # noisy and repeated (a heading echoed twice, empty
            # strings, very short junk fragments). Deduplicate
            # and drop fragments too short to be meaningful.
            seen = set()
            text_parts = []

            for fragment in parsed:

                fragment = clean_text(fragment)

                if not fragment or len(fragment) < 3:
                    continue

                key = fragment.lower()

                if key in seen:
                    continue

                seen.add(key)
                text_parts.append(fragment)

            return ". ".join(
                text_parts
            )

    except (
        ValueError,
        SyntaxError
    ):
        pass

    return description


df["description"] = (
    df["description"]
    .apply(clean_description)
)

print("Description cleaning completed.")

Description cleaning completed.


In [106]:
print("CLEANING RATINGS")

df["rating"] = pd.to_numeric(
    df["rating"],
    errors="coerce"
)

df["rating"] = (
    df["rating"]
    .fillna(0)
)

print(
    df["rating"].describe()
)

CLEANING RATINGS
count    5802.000000
mean        2.877422
std         2.328909
min         0.000000
25%         0.000000
50%         4.000000
75%         5.000000
max         5.000000
Name: rating, dtype: float64


In [107]:
print("EXTRACTING BRANDS")

df["brand"] = df.apply(
    lambda row: extract_brand(
        row["product_category"],
        row["title"]
    ),
    axis=1
)

df["brand"] = (
    df["brand"]
    .fillna("Other")
)

print(
    f"Unique brands: "
    f"{df['brand'].nunique()}"
)

print("\nTop brands:")

print(
    df["brand"]
    .value_counts()
    .head(25)
)

EXTRACTING BRANDS
Unique brands: 30

Top brands:
brand
Samsung      986
Xiaomi       655
Apple        617
Vivo         549
Oppo         461
HP           295
Realme       288
Infinix      285
Tecno        276
Amazon       258
XP Pen       227
Dell         174
Itel         146
Wacom        144
Lenovo       108
Asus          74
Pen Power     54
Zero          45
MSI           38
Mibro         26
Sparx         21
Amazfit       18
Kieslect      17
Dany          10
Acer           8
Name: count, dtype: int64


In [108]:

print("NORMALIZING BRANDS")

BRAND_CORRECTIONS = {
    "Apple Moshi": "Apple",
}


df["brand"] = (
    df["brand"]
    .replace(BRAND_CORRECTIONS)
)


print("Brand corrections applied")


print("\nBrand distribution:")

print(
    df["brand"]
    .value_counts()
    .head(30)
)

NORMALIZING BRANDS
Brand corrections applied

Brand distribution:
brand
Samsung      986
Xiaomi       655
Apple        617
Vivo         549
Oppo         461
HP           295
Realme       288
Infinix      285
Tecno        276
Amazon       258
XP Pen       227
Dell         174
Itel         146
Wacom        144
Lenovo       108
Asus          74
Pen Power     54
Zero          45
MSI           38
Mibro         26
Sparx         21
Amazfit       18
Kieslect      17
Dany          10
Acer           8
Microsoft      7
OnePlus        6
Nokia          4
Huawei         3
Fitbit         2
Name: count, dtype: int64


In [109]:

print("CREATING SUBCATEGORIES")


df["sub_category"] = df.apply(
    lambda row: get_subcategory(
        row["main_category"],
        row["title"],
        row["description"]
    ),
    axis=1
)

print(
    df[
        [
            "main_category",
            "sub_category"
        ]
    ]
    .value_counts()
)

CREATING SUBCATEGORIES
main_category  sub_category         
Mobile Phones  Smartphones              3334
Tablets        Android Tablets           673
Laptops        Laptops                   474
Tablets        Drawing Tablets           371
               iPads                     232
Laptops        MacBooks                  177
               Gaming Laptops            166
               Business Laptops          139
Smart Watches  Smart Watches             129
               Premium Smart Watches      92
Earbuds        AirPods                    15
Name: count, dtype: int64


In [110]:
def convert_price_to_usd(price):
    """
    Convert PKR price to USD.

    Example:
    Rs 6,599 -> 23.57
    """

    if pd.isna(price):
        return None

    price = str(price)

    cleaned = re.sub(
        r"[^\d.]",
        "",
        price
    )

    if not cleaned:
        return None

    try:

        pkr_price = float(
            cleaned
        )

        usd_price = (
            pkr_price
            / PKR_PER_USD
        )

        return round(
            usd_price,
            2
        )

    except ValueError:

        return None

In [111]:
print("CREATING USD PRICE COLUMN")

# Source column in the raw/cleaned dataset
PRICE_COLUMN = "product_price"

# Conversion rate
PKR_PER_USD = 280

# Convert PKR prices to USD
df["price"] = df[PRICE_COLUMN].apply(
    convert_price_to_usd
)

print("USD price column created")

print("\nPrice conversion sample:")

print(
    df[
        ["product_price", "price"]
    ]
    .head(10)
    .to_string(index=False)
)

print(
    f"\nMissing USD prices: "
    f"{df['price'].isna().sum():,}"
)

CREATING USD PRICE COLUMN
USD price column created

Price conversion sample:
product_price  price
    Rs  6,599  23.57
   Rs  10,799  38.57
    Rs  6,999  25.00
   Rs  14,299  51.07
   Rs  19,399  69.28
    Rs  6,299  22.50
   Rs  13,399  47.85
   Rs  12,799  45.71
   Rs  11,799  42.14
   Rs  12,999  46.42

Missing USD prices: 302


In [112]:
print("REMOVING UNREALISTIC PRICES")

# Reasonable upper bounds for this specific product dataset
MAX_PRICE_BY_CATEGORY = {
    "Mobile Phones": 5000,
    "Laptops": 6000,
    "Tablets": 2500,
    "Smart Watches": 1000,
    "Earbuds": 500,
}


def is_valid_price(row):

    price = row["price"]
    category = row["main_category"]

    if pd.isna(price):
        return False

    if price <= 0:
        return False

    maximum = MAX_PRICE_BY_CATEGORY.get(
        category
    )

    if maximum is not None:

        if price > maximum:
            return False

    return True


valid_price_mask = df.apply(
    is_valid_price,
    axis=1
)

invalid_price_count = (
    (~valid_price_mask).sum()
)

print(
    f"Unrealistic/invalid prices removed: "
    f"{invalid_price_count:,}"
)

df = df[
    valid_price_mask
].copy()

print(
    f"Products remaining: "
    f"{len(df):,}"
)

print("\nMaximum price by category:")

print(
    df.groupby(
        "main_category"
    )["price"]
    .agg(["min", "max"])
    .round(2)
)

REMOVING UNREALISTIC PRICES
Unrealistic/invalid prices removed: 1,995
Products remaining: 3,807

Maximum price by category:
                  min      max
main_category                 
Earbuds        139.28   250.00
Laptops          8.93  5933.93
Mobile Phones    1.43  2101.78
Smart Watches   19.64   964.28
Tablets          7.50  1535.71


In [113]:
print("REMOVING INVALID PRODUCTS")

before = len(df)

df = df[
    df["title"].str.len() > 0
].copy()

df = df[
    df["img_url"].str.len() > 0
].copy()

df = df[
    df["description"].str.len() > 0
].copy()

removed = (
    before - len(df)
)

print(
    f"Invalid rows removed: "
    f"{removed:,}"
)

print(
    f"Rows remaining: "
    f"{len(df):,}"
)

REMOVING INVALID PRODUCTS
Invalid rows removed: 27
Rows remaining: 3,780


In [114]:
print("VALIDATING RATINGS")

# Convert rating to numeric
df["rating"] = pd.to_numeric(
    df["rating"],
    errors="coerce"
)


# Find invalid ratings
invalid_rating_mask = (
    df["rating"].isna()
    |
    (df["rating"] < 0)
    |
    (df["rating"] > 5)
)


invalid_rating_count = (
    invalid_rating_mask.sum()
)


print(
    f"Invalid ratings found: "
    f"{invalid_rating_count:,}"
)


# Remove only genuinely invalid ratings
df = df[
    ~invalid_rating_mask
].copy()


print(
    f"Products remaining: "
    f"{len(df):,}"
)


print("\nRating distribution:")

print(
    df["rating"]
    .value_counts()
    .sort_index()
)


print(
    "\nRating 0 is retained "
    "(no rating available)"
)

VALIDATING RATINGS
Invalid ratings found: 0
Products remaining: 3,780

Rating distribution:
rating
0.0    1897
1.0       1
2.0       1
2.3       1
2.5       3
2.8       2
2.9       1
3.0      40
3.1       4
3.2       1
3.3      13
3.4       4
3.5      23
3.6       8
3.7      17
3.8      13
3.9       6
4.0      91
4.1      16
4.2      24
4.3      51
4.4      24
4.5      42
4.6      27
4.7      44
4.8      98
4.9     213
5.0    1115
Name: count, dtype: int64

Rating 0 is retained (no rating available)


In [115]:
print("REMOVING DUPLICATES")

before = len(df)

# First remove duplicates based on product link
if "product_link" in df.columns:

    df = df.drop_duplicates(
        subset=["product_link"],
        keep="first"
    )

# Additional duplicate protection
df = df.drop_duplicates(
    subset=[
        "title",
        "brand",
        "main_category"
    ],
    keep="first"
)

duplicates_removed = (
    before - len(df)
)

print(
    f"Duplicates removed: "
    f"{duplicates_removed:,}"
)

print(
    f"Clean products available: "
    f"{len(df):,}"
)

REMOVING DUPLICATES
Duplicates removed: 986
Clean products available: 2,794


In [116]:

print("CLEAN DATASET INSPECTION")


print(
    f"Clean products available: "
    f"{len(df):,}"
)

print("\nMain categories:")

print(
    df["main_category"]
    .value_counts()
)

print("\nBrands:")

print(
    df["brand"]
    .value_counts()
    .head(20)
)

print("\nMissing values:")

print(
    df[
        [
            "title",
            "description",
            "img_url",
            "brand",
            "rating",
            "main_category",
            "sub_category",
            "price"
        ]
    ]
    .isna()
    .sum()
)

CLEAN DATASET INSPECTION
Clean products available: 2,794

Main categories:
main_category
Mobile Phones    1801
Laptops           715
Smart Watches     145
Tablets           129
Earbuds             4
Name: count, dtype: int64

Brands:
brand
Samsung    428
Xiaomi     337
Vivo       288
Apple      255
Oppo       235
HP         228
Tecno      159
Infinix    155
Realme     147
Dell       108
Lenovo     100
Itel        88
Asus        74
MSI         35
Zero        33
XP Pen      19
Amazon      16
Sparx       15
Amazfit     15
Mibro       13
Name: count, dtype: int64

Missing values:
title            0
description      0
img_url          0
brand            0
rating           0
main_category    0
sub_category     0
price            0
dtype: int64


In [117]:
if len(df) < TARGET_PRODUCTS:

    raise ValueError(
        f"Only {len(df):,} clean products "
        f"are available, but "
        f"{TARGET_PRODUCTS:,} are required."
    )

print(
    f"Enough products available: "
    f"{len(df):,}"
)

print(
    f"Target sample size: "
    f"{TARGET_PRODUCTS:,}"
)

Enough products available: 2,794
Target sample size: 1,000


In [118]:
print("CALCULATING CATEGORY SAMPLE SIZES")

category_counts = (
    df["main_category"]
    .value_counts()
)

category_proportions = (
    category_counts
    / len(df)
)

sample_sizes = (
    category_proportions
    * TARGET_PRODUCTS
).round().astype(int)


# Make sure every category gets at least 1 product
sample_sizes = sample_sizes.clip(
    lower=1
)


# Correct rounding so total equals exactly 1000
difference = (
    TARGET_PRODUCTS
    - sample_sizes.sum()
)

if difference != 0:

    largest_category = (
        category_counts
        .idxmax()
    )

    sample_sizes[
        largest_category
    ] += difference


print("\nAvailable products:")

print(category_counts)

print("\nProducts to sample:")

print(sample_sizes)

print(
    f"\nTotal sample size: "
    f"{sample_sizes.sum()}"
)

CALCULATING CATEGORY SAMPLE SIZES

Available products:
main_category
Mobile Phones    1801
Laptops           715
Smart Watches     145
Tablets           129
Earbuds             4
Name: count, dtype: int64

Products to sample:
main_category
Mobile Phones    645
Laptops          256
Smart Watches     52
Tablets           46
Earbuds            1
Name: count, dtype: int64

Total sample size: 1000


test cell

In [119]:

print("ACCESSORY AUDIT")


audit_keywords = [
    "iphone case",
    "phone case",
    "mobile case",
    "back cover",
    "screen protector",
    "screen shield",
    "tempered glass",
    "lens protector",
    "charging cable",
    "data cable",
    "usb cable",
    "type-c cable",
    "lightning cable",
    "power bank",
    "phone holder",
    "mobile holder",
    "car mount",
    "laptop bag",
    "laptop sleeve",
    "laptop stand",
    "laptop cooler",
    "cooling pad",
    "mouse pad",
    "keyboard cover",
]


pattern = "|".join(
    re.escape(keyword)
    for keyword in audit_keywords
)

possible_accessories = df[
    df["title"]
    .str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    )
]

print(
    f"Potential accessory records: "
    f"{len(possible_accessories):,}"
)

if len(possible_accessories) > 0:

    print(
        possible_accessories[
            [
                "title",
                "main_category",
                "brand",
                "price"
            ]
        ].head(50).to_string(
            index=False
        )
    )

ACCESSORY AUDIT
Potential accessory records: 0


In [120]:

print("PRICE AUDIT")


print(
    df[
        [
            "title",
            "main_category",
            "price"
        ]
    ]
    .sort_values(
        "price",
        ascending=False
    )
    .head(20)
    .to_string(
        index=False
    )
)

PRICE AUDIT
                                                                          title main_category   price
                    Asus ROG Zephyrus M16 GU604VY-NM061W i9-13900H 32GB 2TB SSD       Laptops 5933.93
                   Asus Rog Strix Scar 18 G834JY-N6064W i9-13980HX 32GB 2TB SSD       Laptops 5622.85
                         Apple MacBook Pro 16.2" M2 Max 64GB RAM 2TB Space Grey       Laptops 5214.28
       Apple MacBook Pro Z1AH000VQ M3 Max 16 Inch 64GB 4TB SSD Space Black 2023       Laptops 5160.71
                    MSi Titan GT77HX 13VH i9-13980HX 64GB 2TB SSD Gaming Laptop       Laptops 5076.43
          Apple MacBook Pro Z1AH000VV M3 Max 16 Inch 128GB 1TB Space Black 2023       Laptops 4946.43
               MSi Stealth 17 Studio A13VH i9-13900H 32GB 2TB SSD Gaming Laptop       Laptops 4837.14
                Apple MacBook Pro 16 M3 Max Z1AF0001AG 64GB 2TB SSD Space Black       Laptops 4564.28
            Asus ROG Zephyrus Duo 16 GX650RX-L0192W Ryzen 9 6900HX 32G

test cells end

In [121]:

print("SELECTING 1,000 PRODUCTS")


sampled_parts = []

for category, sample_size in sample_sizes.items():

    category_df = df[
        df["main_category"]
        == category
    ]

    sampled_category = (
        category_df
        .sample(
            n=sample_size,
            random_state=RANDOM_STATE
        )
    )

    sampled_parts.append(
        sampled_category
    )


final_df = pd.concat(
    sampled_parts,
    ignore_index=True
)


# Shuffle the final dataset
final_df = (
    final_df
    .sample(
        frac=1,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)


print(
    f"Selected products: "
    f"{len(final_df):,}"
)

SELECTING 1,000 PRODUCTS
Selected products: 1,000


In [122]:
def create_recommendation_text(row):

    parts = []

    title = clean_text(
        row["title"]
    )

    description = clean_text(
        row["description"]
    )

    brand = clean_text(
        row["brand"]
    )

    main_category = clean_text(
        row["main_category"]
    )

    sub_category = clean_text(
        row["sub_category"]
    )

    if title:
        parts.append(title)

    if brand:
        parts.append(brand)

    if main_category:
        parts.append(main_category)

    if sub_category:
        parts.append(sub_category)

    if description:
        parts.append(description)

    return re.sub(
        r"\s+",
        " ",
        " ".join(parts)
    ).strip()

In [123]:

print("CREATING RECOMMENDATION TEXT")

def create_recommendation_text(row):

    parts = []

    # Product title
    title = str(
        row["title"]
    ).strip()

    if title:
        parts.append(title)

    # Brand
    brand = str(
        row["brand"]
    ).strip()

    if brand:
        parts.append(brand)

    # Main category
    main_category = str(
        row["main_category"]
    ).strip()

    if main_category:
        parts.append(main_category)

    # Subcategory
    sub_category = str(
        row["sub_category"]
    ).strip()

    if sub_category:
        parts.append(sub_category)

    # Description
    description = str(
        row["description"]
    ).strip()

    if description:
        parts.append(description)

    return " ".join(parts)


final_df["recommendation_text"] = final_df.apply(
    create_recommendation_text,
    axis=1
)


print(
    "recommendation_text created"
)

print(
    f"Records processed: {len(final_df):,}"
)


print("\nSample recommendation text:\n")

print(
    final_df[
        [
            "title",
            "brand",
            "main_category",
            "sub_category",
            "recommendation_text"
        ]
    ]
    .head(5)
    .to_string(index=False)
)

CREATING RECOMMENDATION TEXT
recommendation_text created
Records processed: 1,000

Sample recommendation text:

                                                                          title   brand main_category   sub_category                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         recommendation_text
                                                      Tecno Spark Go 1 4GB 64GB   Tecno Mobile Phones  

In [124]:

print("CREATING PRODUCT IDs")


final_df.insert(
    0,
    "product_id",
    [
        f"P{i:04d}"
        for i in range(
            1,
            len(final_df) + 1
        )
    ]
)

print(
    "First 10 IDs:"
)

print(
    final_df[
        "product_id"
    ].head(10).tolist()
)

print(
    "\nLast 10 IDs:"
)

print(
    final_df[
        "product_id"
    ].tail(10).tolist()
)

CREATING PRODUCT IDs
First 10 IDs:
['P0001', 'P0002', 'P0003', 'P0004', 'P0005', 'P0006', 'P0007', 'P0008', 'P0009', 'P0010']

Last 10 IDs:
['P0991', 'P0992', 'P0993', 'P0994', 'P0995', 'P0996', 'P0997', 'P0998', 'P0999', 'P1000']


In [125]:
print("SELECTING FINAL COLUMNS")


final_columns = [
    "product_id",
    "title",
    "description",
    "img_url",
    "recommendation_text",
    "brand",
    "rating",
    "main_category",
    "sub_category",
    "price"
]

final_df = final_df[
    final_columns
].copy()

print("\nFinal columns:")

for i, column in enumerate(
    final_df.columns,
    start=1
):

    print(
        f"{i}. {column}"
    )

SELECTING FINAL COLUMNS

Final columns:
1. product_id
2. title
3. description
4. img_url
5. recommendation_text
6. brand
7. rating
8. main_category
9. sub_category
10. price


In [126]:
print("FINAL DATASET VALIDATION")

# ROW COUNT

assert len(final_df) == 1000, (
    f"Expected 1000 products, "
    f"got {len(final_df)}"
)

print(" Exactly 1,000 products")

# COLUMNS

assert list(final_df.columns) == [
    "product_id",
    "title",
    "description",
    "img_url",
    "recommendation_text",
    "brand",
    "rating",
    "main_category",
    "sub_category",
    "price"
]

print("Correct 10 columns")

# PRODUCT IDS

assert final_df["product_id"].is_unique

assert final_df["product_id"].iloc[0] == "P0001"

assert final_df["product_id"].iloc[-1] == "P1000"

print("Product IDs P0001 → P1000")

# MISSING VALUES

assert not final_df.isna().any().any()

print("No missing values")


# PRICE

assert (
    final_df["price"] > 0
).all()

assert (
    final_df["price"] <= 6000
).all()

print("Prices are positive and within valid range")

# RATINGS


assert (
    final_df["rating"] >= 0
).all()

assert (
    final_df["rating"] <= 5
).all()

print("Ratings are between 0 and 5")

# ACCESSORY CHECK

assert not final_df[
    "title"
].str.contains(
    r"iphone case|phone case|mobile case|back cover|"
    r"screen protector|tempered glass|"
    r"charging cable|data cable|usb cable|"
    r"power bank|phone holder|mobile holder|"
    r"laptop bag|laptop sleeve|laptop stand",
    case=False,
    na=False,
    regex=True
).any()

print("No obvious accessories")


# RECOMMENDATION TEXT

assert (
    final_df["recommendation_text"]
    .str.len() > 0
).all()

print("Recommendation text exists")


# FINAL RESULT
print("VALIDATION PASSED")

FINAL DATASET VALIDATION
 Exactly 1,000 products
Correct 10 columns
Product IDs P0001 → P1000
No missing values
Prices are positive and within valid range
Ratings are between 0 and 5
No obvious accessories
Recommendation text exists
VALIDATION PASSED


In [127]:

print("FINAL DATASET REPORT")

print(
    f"\nFinal rows: "
    f"{len(final_df):,}"
)

print(
    f"Final columns: "
    f"{len(final_df.columns)}"
)


print("\nMAIN CATEGORY DISTRIBUTION:")

print(
    final_df[
        "main_category"
    ].value_counts()
)


print("\nBRAND DISTRIBUTION:")

print(
    final_df[
        "brand"
    ].value_counts()
    .head(30)
)


print("\nSUBCATEGORY DISTRIBUTION:")

print(
    final_df[
        "sub_category"
    ].value_counts()
    .head(30)
)


print("\nMISSING VALUES:")

print(
    final_df.isna().sum()
)


print("\nPRICE STATISTICS:")

print(
    final_df["price"].describe()
)

FINAL DATASET REPORT

Final rows: 1,000
Final columns: 10

MAIN CATEGORY DISTRIBUTION:
main_category
Mobile Phones    645
Laptops          256
Smart Watches     52
Tablets           46
Earbuds            1
Name: count, dtype: int64

BRAND DISTRIBUTION:
brand
Samsung      152
Xiaomi       109
Vivo          97
Oppo          93
Apple         92
HP            83
Tecno         62
Infinix       61
Realme        56
Dell          44
Lenovo        38
Itel          28
Asus          27
Zero          13
MSI            9
Mibro          6
XP Pen         6
Amazon         5
Amazfit        4
Sparx          4
Microsoft      2
Wacom          2
Kieslect       2
Dany           2
Fitbit         1
Pen Power      1
Nokia          1
Name: count, dtype: int64

SUBCATEGORY DISTRIBUTION:
sub_category
Smartphones              645
Laptops                  130
Gaming Laptops            45
MacBooks                  42
Business Laptops          39
Smart Watches             32
Android Tablets           28
Premium Smart

In [128]:
# Save the final recommendation dataset.
# The original raw dataset remains untouched.

output_path = "../processed/products.csv"

final_df.to_csv(
    output_path,
    index=False
)

print("Final dataset saved successfully.")
print("Location:", output_path)

Final dataset saved successfully.
Location: ../processed/products.csv
